# FFN Bot Detector - TwiBot-22

Feed-forward neural network on TF-IDF features.

**Input:** per-user document = concatenated tweets (up to 20) + profile description  
**Features:** TF-IDF (5000 dims), fit on train split only  
**Architecture:** 5000 -> 512 -> 256 -> 128 -> 2  
**Labels:** human = 0, bot = 1

In [2]:
!pip install ijson scipy scikit-learn

In [69]:
import os, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import ijson
from tqdm.auto import tqdm
from scipy.sparse import save_npz, load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, classification_report
)

## Setup

In [70]:
DATA_DIR = Path(r'G:\.shortcut-targets-by-id\1YwiOUwtl8pCd2GD97Q_WEzwEUtSPoxFs\TwiBot-22')
assert (DATA_DIR / 'label.csv').exists(), f'label.csv not found in {DATA_DIR}'

WORK_DIR = Path(r'C:\Users\notdu\OneDrive\Desktop\CS4120\SpotTheBot\work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

MAX_TWEETS_PER_USER = 20
TFIDF_FEATURES      = 5000

BATCH_SIZE = 512
EPOCHS     = 50
LR         = 5e-4
WD         = 1e-4
PATIENCE   = 5

Device: cuda


## Load labels and splits

In [71]:
label_df = pd.read_csv(DATA_DIR / 'label.csv')
split_df  = pd.read_csv(DATA_DIR / 'split.csv')

users_index_to_uid = list(label_df['id'])
uid_to_users_index = {u: i for i, u in enumerate(users_index_to_uid)}
N_USERS = len(users_index_to_uid)

# human=0, bot=1
label_all = torch.tensor(
    [0 if l == 'human' else 1 for l in label_df['label']],
    dtype=torch.long
)

split_map = dict(zip(split_df['id'], split_df['split']))
split_idx = {'train': [], 'val': [], 'test': []}
for uid, idx in uid_to_users_index.items():
    s = split_map.get(uid)
    if s in split_idx:
        split_idx[s].append(idx)

for k, v in split_idx.items():
    print(f'{k}: {len(v):,} users')

train: 700,000 users
val: 200,000 users
test: 100,000 users


## Load user descriptions

In [ ]:
descriptions = [''] * N_USERS

with open(DATA_DIR / 'user.json', 'rb') as f:
    for u in tqdm(ijson.items(f, 'item'), desc='user.json'):
        uid = u.get('id')
        idx = uid_to_users_index.get(uid)
        if idx is None:
            continue
        descriptions[idx] = u.get('description') or ''

print(f'Non-empty descriptions: {sum(1 for d in descriptions if d):,}')

## Load tweets

In [7]:
edge_iter = pd.read_csv(DATA_DIR / 'edge.csv', chunksize=2_000_000)
tid_to_user = {}
for chunk in tqdm(edge_iter, desc='edge.csv'):
    posts = chunk[chunk['relation'] == 'post']
    for src, tgt in zip(posts['source_id'].values, posts['target_id'].values):
        uidx = uid_to_users_index.get(src)
        if uidx is None:
            continue
        if tgt not in tid_to_user:
            tid_to_user[tgt] = uidx

user_tweets = [[] for _ in range(N_USERS)]
for tf in tqdm(sorted(DATA_DIR.glob('tweet_*.json')), desc='tweet files'):
    with open(tf, 'rb') as f:
        for t in tqdm(ijson.items(f, 'item'), desc=tf.name, leave=False):
            tid  = t.get('id')
            uidx = tid_to_user.get(tid)
            if uidx is None:
                continue
            if len(user_tweets[uidx]) >= MAX_TWEETS_PER_USER:
                continue
            text = t.get('text') or ''
            if text:
                user_tweets[uidx].append(text)

del tid_to_user; gc.collect()
print(f'Users with >=1 tweet: {sum(1 for t in user_tweets if t):,}')
np.save(WORK_DIR / 'user_tweets_dict.npy', np.array(user_tweets, dtype=object), allow_pickle=True)

edge.csv: 86it [03:54,  2.73s/it]
tweet files: 100%|██████████| 9/9 [42:19<00:00, 282.22s/it]


Users with >=1 tweet: 933,872


## TF-IDF features

Each user's tweets and profile description are joined into one document. The vectorizer is fit on training users only, then applied to all splits. Saved to disk so this only needs to run once.

In [8]:
all_texts = []
for i in range(N_USERS):
    tweets_str = ' '.join(user_tweets[i])
    doc = (tweets_str + ' ' + descriptions[i]).strip()
    all_texts.append(doc if doc else ' ')

del user_tweets, descriptions; gc.collect()

train_texts = [all_texts[i] for i in split_idx['train']]
vectorizer = TfidfVectorizer(max_features=TFIDF_FEATURES, stop_words='english', min_df=2)
vectorizer.fit(train_texts)

# Store as sparse matrix -- 1M x 5000 dense would be ~20 GB
X_all_sparse = vectorizer.transform(all_texts).astype(np.float32)
save_npz(WORK_DIR / 'tfidf_all.npz', X_all_sparse)

print(f'TF-IDF matrix: {X_all_sparse.shape}, nnz={X_all_sparse.nnz:,}')
del all_texts, train_texts; gc.collect()

TF-IDF matrix: (1000000, 5000), nnz=80,065,792


0

### Reload from disk

In [80]:
X_all_sparse = load_npz(WORK_DIR / 'tfidf_all.npz')
print(f'Loaded TF-IDF matrix: {X_all_sparse.shape}')

Loaded TF-IDF matrix: (1000000, 5000)


## Dataset and DataLoaders

In [81]:
class TwibotFFNDataset(Dataset):
    def __init__(self, split_name):
        idx = np.array(split_idx[split_name])
        self.X      = X_all_sparse[idx]
        self.labels = label_all[idx]

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        x = torch.tensor(self.X[i].toarray().flatten(), dtype=torch.float32)
        return x, self.labels[i]


train_loader = DataLoader(TwibotFFNDataset('train'), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TwibotFFNDataset('val'),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TwibotFFNDataset('test'),  batch_size=BATCH_SIZE, shuffle=False)

print(f'Batches -- train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')

Batches -- train: 1368, val: 391, test: 196


## Model

In [94]:
class FF_Net(nn.Module):
    def __init__(self, input_dim, hidden_dims=(512, 256, 128), dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 2))  # human=0, bot=1
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model     = FF_Net(input_dim=TFIDF_FEATURES).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
loss_fn   = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 5.0]).to(DEVICE))

print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

FF_Net(
  (net): Sequential(
    (0): Linear(in_features=5000, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=128, out_features=2, bias=True)
  )
)
Trainable parameters: 2,724,994


## Metrics

In [95]:
def report(preds_auc, preds, labels, tag):
    print(f'[{tag}] ACC={accuracy_score(labels, preds):.4f} '
          f'F1={f1_score(labels, preds):.4f} '
          f'ROC={roc_auc_score(labels, preds_auc):.4f} '
          f'P={precision_score(labels, preds):.4f} '
          f'R={recall_score(labels, preds):.4f}')

## Training

In [96]:
def run(loader, train=False):
    model.train(train)
    preds, aucs, labs, tot = [], [], [], 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            out  = model(x)
            loss = loss_fn(out, y)
        if train:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot  += loss.item() * y.size(0)
        preds.append(out.argmax(-1).cpu().numpy())
        aucs.append(F.softmax(out, dim=-1)[:, 1].detach().cpu().numpy())
        labs.append(y.cpu().numpy())
    return (
        tot / len(loader.dataset),
        np.concatenate(aucs),
        np.concatenate(preds),
        np.concatenate(labs),
    )


best_val_acc      = 0.0
patience_counter = 0

for ep in range(EPOCHS):
    tr_loss, tr_auc, tr_p, tr_y = run(train_loader, train=True)
    va_loss, va_auc, va_p, va_y = run(val_loader)

    print(f'epoch {ep:02d}  train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}')
    report(tr_auc, tr_p, tr_y, 'train')
    report(va_auc, va_p, va_y, 'val  ')

    val_acc = accuracy_score(va_y, va_p)
    if val_acc > best_val_acc:
        best_val_acc      = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), WORK_DIR / 'ffn_best.pt')
        print(f'  -> new best val ACC: {best_val_acc:.4f} (saved)')
    else:
        patience_counter += 1
        print(f'  no improvement ({patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {ep}.')
            break

epoch 00  train_loss=0.5105  val_loss=0.8448
[train] ACC=0.8754 F1=0.3257 ROC=0.7636 P=0.2818 R=0.3860
[val  ] ACC=0.6638 F1=0.3655 ROC=0.6063 P=0.3869 R=0.3463
  -> new best val ACC: 0.6638 (saved)
epoch 01  train_loss=0.4919  val_loss=0.9271
[train] ACC=0.8752 F1=0.3512 ROC=0.7861 P=0.2954 R=0.4331
[val  ] ACC=0.6851 F1=0.3383 ROC=0.6076 P=0.4100 R=0.2880
  -> new best val ACC: 0.6851 (saved)
epoch 02  train_loss=0.4792  val_loss=0.9249
[train] ACC=0.8727 F1=0.3654 ROC=0.8008 P=0.2988 R=0.4702
[val  ] ACC=0.6795 F1=0.3545 ROC=0.6089 P=0.4057 R=0.3148
  no improvement (1/5)
epoch 03  train_loss=0.4603  val_loss=0.9404
[train] ACC=0.8701 F1=0.3856 ROC=0.8211 P=0.3055 R=0.5226
[val  ] ACC=0.6679 F1=0.3723 ROC=0.6095 P=0.3948 R=0.3523
  no improvement (2/5)
epoch 04  train_loss=0.4293  val_loss=1.0335
[train] ACC=0.8705 F1=0.4172 ROC=0.8495 P=0.3214 R=0.5944
[val  ] ACC=0.6687 F1=0.3713 ROC=0.6074 P=0.3954 R=0.3500
  no improvement (3/5)
epoch 05  train_loss=0.3909  val_loss=1.0706
[trai

## Test evaluation

In [97]:
model.load_state_dict(torch.load(WORK_DIR / 'ffn_best.pt'))
te_loss, te_auc, te_p, te_y = run(test_loader)

print('\n=== best-val checkpoint on test ===')
report(te_auc, te_p, te_y, 'test ')
print()
print(classification_report(te_y, te_p, target_names=['Human', 'Bot']))
print('Confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(te_y, te_p))

C:\Users\notdu\AppData\Local\Temp\ipykernel_28480\3620965521.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(WORK_DIR / 'ffn_best.pt'))



=== best-val checkpoint on test ===
[test ] ACC=0.6978 F1=0.4198 ROC=0.6523 P=0.4828 R=0.3713

              precision    recall  f1-score   support

       Human       0.76      0.83      0.80     70556
         Bot       0.48      0.37      0.42     29444

    accuracy                           0.70    100000
   macro avg       0.62      0.60      0.61    100000
weighted avg       0.68      0.70      0.68    100000

Confusion matrix (rows=true, cols=pred):
[[58845 11711]
 [18511 10933]]


## Confidence analysis

In [98]:
model.eval()
probs_all, preds_all, labels_all = [], [], []

with torch.no_grad():
    for x, y in DataLoader(TwibotFFNDataset('test'), batch_size=256):
        logits = model(x.to(DEVICE))
        p = F.softmax(logits, dim=-1).cpu().numpy()
        probs_all.append(p)
        preds_all.append(p.argmax(1))
        labels_all.append(y.numpy())

probs_all  = np.concatenate(probs_all)   # [N, 2], col 1 = p(bot)
preds_all  = np.concatenate(preds_all)
labels_all = np.concatenate(labels_all)
confidences = probs_all.max(axis=1)
bot_scores  = probs_all[:, 1]
correct     = (preds_all == labels_all)

print(f'Mean confidence:          {confidences.mean():.4f}')
print(f'Median confidence:        {np.median(confidences):.4f}')
print(f'Std confidence:           {confidences.std():.4f}')
print(f'Mean conf when correct:   {confidences[correct].mean():.4f}')
print(f'Mean conf when wrong:     {confidences[~correct].mean():.4f}')

buckets = [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 0.95), (0.95, 1.01)]
print(f'\n{"confidence bucket":<20} {"count":>7} {"accuracy":>10}')
for lo, hi in buckets:
    mask = (confidences >= lo) & (confidences < hi)
    if mask.sum() > 0:
        print(f'  [{lo:.2f}, {hi:.2f})  {mask.sum():>7}  {correct[mask].mean():>10.4f}')

Mean confidence:          0.7430
Median confidence:        0.7508
Std confidence:           0.1263
Mean conf when correct:   0.7604
Mean conf when wrong:     0.7029

confidence bucket      count   accuracy
  [0.50, 0.60)    17058      0.5494
  [0.60, 0.70)    19670      0.6346
  [0.70, 0.80)    27802      0.6970
  [0.80, 0.90)    22599      0.7856
  [0.90, 0.95)    10015      0.8354
  [0.95, 1.01)     2856      0.8491


## Save predictions

In [ ]:
test_user_ids = [users_index_to_uid[i] for i in split_idx['test']]

pd.DataFrame({
    'user_id':    test_user_ids,
    'true':       labels_all,
    'pred':       preds_all,
    'p_bot':      bot_scores,
    'confidence': confidences,
    'correct':    correct.astype(int),
}).to_csv(WORK_DIR / 'test_predictions.csv', index=False)

print(f'Saved {WORK_DIR}/test_predictions.csv')

Saved C:\Users\notdu\OneDrive\Desktop\CS4120\SpotTheBot\work/test_predictions.csv
